# SatMesh Training — Google Colab
Runtime → Change runtime type → **T4 GPU**

In [ ]:
# Cell 1: Mount Drive (checkpoints survive session death)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_CKPT = '/content/drive/MyDrive/satmesh_checkpoints'
import os; os.makedirs(DRIVE_CKPT, exist_ok=True)
print('Drive mounted. Checkpoints →', DRIVE_CKPT)

In [ ]:
# Cell 2: GPU check
import torch
assert torch.cuda.is_available(), 'No GPU! Change runtime type.'
cap = torch.cuda.get_device_capability()
print(f'GPU: {torch.cuda.get_device_name(0)}  capability: {cap}')
assert cap[0] >= 7, f'GPU too old (sm_{cap[0]}{cap[1]}), need sm_70+'

In [ ]:
# Cell 3: Kaggle credentials (upload your kaggle.json)
# Get kaggle.json from: https://www.kaggle.com/settings → API → Create New Token
from google.colab import files
print('Upload your kaggle.json now...')
uploaded = files.upload()  # select kaggle.json
import os, shutil
os.makedirs(os.path.expanduser('~/.config/kaggle'), exist_ok=True)
shutil.copy('kaggle.json', os.path.expanduser('~/.config/kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.config/kaggle/kaggle.json'), 0o600)
print('Kaggle credentials set.')

In [ ]:
# Cell 4: Clone private repo
# Create token: GitHub → Settings → Developer settings → Personal access tokens → Fine-grained
# Permissions: Contents (read)
import os
GH_TOKEN = 'ghp_REPLACE_WITH_YOUR_TOKEN'  # <-- paste your token here
REPO = f'https://{GH_TOKEN}@github.com/SahilKumar75/SatMesh.git'
ROOT = '/content/SatMesh'

if os.path.exists(os.path.join(ROOT, '.git')):
    os.system(f'git -C {ROOT} pull origin main')
else:
    os.system(f'git clone {REPO} {ROOT}')
os.chdir(ROOT)
print('Repo ready at', ROOT)

In [ ]:
# Cell 5: Install deps
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'segmentation-models-pytorch', 'rasterio', 'albumentations',
    'kagglehub', '-q'], check=True)
print('Deps installed.')

In [ ]:
# Cell 6: Download DeepGlobe dataset via kagglehub
import kagglehub, glob
root = kagglehub.dataset_download('balraj98/deepglobe-road-extraction-dataset')
cands = glob.glob(f'{root}/**/train', recursive=True)
DATA = next((c for c in cands if glob.glob(f'{c}/*_sat.jpg')), None)
assert DATA, f'No train/*_sat.jpg under {root}'
print(f'Dataset: {DATA} | {len(glob.glob(DATA+"/*_sat.jpg"))} tiles')

In [ ]:
# Cell 7: Train — checkpoints saved to Drive every epoch
# Resume: if stage1.pth exists in Drive it will be picked up automatically
import subprocess, sys, os, shutil

CKPT_LOCAL = '/content/SatMesh/checkpoints/segformer_b4'
os.makedirs(CKPT_LOCAL, exist_ok=True)

# Restore checkpoint from Drive if exists (resume after session death)
drive_ckpt = f'{DRIVE_CKPT}/stage1.pth'
local_ckpt = f'{CKPT_LOCAL}/stage1.pth'
if os.path.exists(drive_ckpt) and not os.path.exists(local_ckpt):
    shutil.copy(drive_ckpt, local_ckpt)
    print('Restored checkpoint from Drive.')

cmd = [
    sys.executable, 'scripts/train.py',
    '--deepglobe',      DATA,
    '--encoder',        'mit_b4',
    '--model',          'segformer',
    '--epochs',         '20',   # 20 epochs ~ fits T4 session
    '--batch',          '4',
    '--grad-checkpoint',
    '--out',            CKPT_LOCAL,
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd)

# Copy checkpoint to Drive
for fname in ['stage1.pth', 'segformer_india.pth']:
    src = os.path.join(CKPT_LOCAL, fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(DRIVE_CKPT, fname))
        print(f'Saved {fname} → Drive')

In [ ]:
# Cell 8: Download final checkpoint
from google.colab import files
import os
for fname in ['stage1.pth', 'segformer_india.pth']:
    path = f'{DRIVE_CKPT}/{fname}'
    if os.path.exists(path):
        files.download(path)
        print(f'Downloading {fname}')